# DocBank Dataset Download and Preparation for YOLO

This notebook downloads DocBank dataset from HuggingFace and converts it to YOLO format for text, formula, and image detection.

**Dataset Info:**
- 500K document images (400K train, 50K val, 50K test)
- Classes: text, formula, image
- Total size: ~55GB (images) + annotations

**⚠️ Important:** Full dataset is very large. This notebook includes options for downloading subsets.

## Step 1: Install Dependencies

In [1]:
%pip install -q huggingface_hub ultralytics pillow tqdm matplotlib

Note: you may need to restart the kernel to use updated packages.


## Step 2: Set Local Working Directory (Optional but Recommended)

Choose a local folder where datasets and training artifacts will be stored.
By default, this notebook creates and uses a `DocBank` directory near your current notebook working directory.

In [2]:
from pathlib import Path
import os

# Set a local working directory
PROJECT_ROOT = Path.cwd() / "DocBank"
PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
os.chdir(PROJECT_ROOT)

print(f"Working directory: {Path.cwd()}")

Working directory: /Users/roman/Documents/docbank conv/DocBank


## Step 3: Download Dataset from HuggingFace

### Option A: Download MSCOCO Annotations (RECOMMENDED - Small, Easy)

The MSCOCO format annotations are only 209MB and much easier to work with.

In [4]:
from huggingface_hub import hf_hub_download
import zipfile
import os

# Download MSCOCO format annotations
print("Downloading MSCOCO annotations...")
annotation_zip = hf_hub_download(
    repo_id="liminghao1630/DocBank",
    filename="MSCOCO_Format_Annotation.zip",
    repo_type="dataset",
    local_dir="."
)

# Extract annotations
print("Extracting annotations...")
with zipfile.ZipFile(annotation_zip, 'r') as zip_ref:
    zip_ref.extractall('annotations')

print("✓ Annotations downloaded and extracted")
print(f"Location: {os.getcwd()}/annotations/")

Extracting annotations...
✓ Annotations downloaded and extracted
Location: /Users/roman/Documents/docbank conv/DocBank/annotations/


### Option B: Download Images

**Choose one of the following:**

1. **Full Dataset (55GB)** - All 10 parts
2. **Subset** - Download only a few parts for testing
3. **Stream from HuggingFace** - Don't download, load on-the-fly (slower training)

In [4]:
# Option B1: Download ALL image parts (⚠️ 55GB total, takes ~1-2 hours)
# Uncomment to use:

# from huggingface_hub import hf_hub_download
# from pathlib import Path

# print("Downloading all 10 image archive parts (this will take a while)...")
# for i in range(1, 11):
#     filename = f"DocBank_500K_ori_img.zip.{i:03d}"
#     print(f"Downloading part {i}/10: {filename}")
#     hf_hub_download(
#         repo_id="liminghao1630/DocBank",
#         filename=filename,
#         repo_type="dataset",
#         local_dir=".",
#         resume_download=True
#     )

# print("\nAll parts downloaded.")
# print("Now run the main extraction cell below (Step 3 Option B2) with NUM_PARTS = 10")

In [5]:
from huggingface_hub import hf_hub_download
from pathlib import Path
import subprocess
import glob

NUM_PARTS = 2

# Download parts
print("Downloading parts...")
for i in range(1, NUM_PARTS + 1):
    filename = f"DocBank_500K_ori_img.zip.{i:03d}"
    print(f"[{i}/{NUM_PARTS}] Downloading {filename}...")
    
    hf_hub_download(
        repo_id="liminghao1630/DocBank",
        filename=filename,
        repo_type="dataset",
        local_dir="."
    )
    print(f"✓ Part {i} downloaded")

# Combine using zip -F (Fix command)
print("\nCombining archives with zip -F...")
result = subprocess.run(
    ["zip", "-F", "DocBank_500K_ori_img.zip.001", "--out", "combined.zip"],
    capture_output=True,
    text=True
)

if result.returncode != 0:
    print("Error:", result.stderr)
    print("\nTrying alternative method with 7zip...")
    # Install 7zip via homebrew if needed
    subprocess.run(["brew", "install", "p7zip"], capture_output=True)
    subprocess.run(["7z", "x", "DocBank_500K_ori_img.zip.001", "-o./images/"])
else:
    # Extract combined zip
    print("Extracting combined.zip...")
    subprocess.run(["unzip", "-q", "combined.zip", "-d", "images/"])

# Verify
imgs = glob.glob("images/**/*.jpg", recursive=True)
print(f"\n✓ SUCCESS! {len(imgs)} images extracted")
# from huggingface_hub import hf_hub_download
# from pathlib import Path
# import glob
# import shutil
# import zipfile

# NUM_PARTS = 2
# ARCHIVE_BASENAME = "DocBank_500K_ori_img.zip"
# CLEANUP_AFTER_EXTRACT = False  # Set True to remove downloaded archives after extraction

# def combine_split_zip(parts, output_zip):
#     """Combine .zip.001/.zip.002/... parts into a single .zip file."""
#     with open(output_zip, "wb") as out_f:
#         for part in parts:
#             print(f"Appending {part.name}...")
#             with open(part, "rb") as part_f:
#                 shutil.copyfileobj(part_f, out_f)

# def extract_zip(zip_path, target_dir):
#     target_dir.mkdir(parents=True, exist_ok=True)
#     with zipfile.ZipFile(zip_path, "r") as zip_ref:
#         zip_ref.extractall(target_dir)

# # Download parts
# print("Downloading parts...")
# downloaded_parts = []
# for i in range(1, NUM_PARTS + 1):
#     filename = f"{ARCHIVE_BASENAME}.{i:03d}"
#     print(f"[{i}/{NUM_PARTS}] Downloading {filename}...")
#     part_path = hf_hub_download(
#         repo_id="liminghao1630/DocBank",
#         filename=filename,
#         repo_type="dataset",
#         local_dir=".",
#         resume_download=True,
#     )
#     downloaded_parts.append(Path(part_path))
#     print(f"✓ Part {i} downloaded")

# # Combine and extract
# downloaded_parts = sorted(downloaded_parts, key=lambda p: p.name)
# combined_zip_path = Path(ARCHIVE_BASENAME)
# print("\nCombining split archives...")
# combine_split_zip(downloaded_parts, combined_zip_path)

# print("Extracting ZIP...")
# extract_zip(combined_zip_path, Path("images"))

# # Verify
# imgs = glob.glob("images/**/*.jpg", recursive=True) + glob.glob("images/**/*.png", recursive=True)
# print(f"\n✓ SUCCESS! {len(imgs)} images extracted to images/")

# # Optional cleanup
# if CLEANUP_AFTER_EXTRACT:
#     for part in downloaded_parts:
#         part.unlink(missing_ok=True)
#     combined_zip_path.unlink(missing_ok=True)
#     print("✓ Cleaned up archive files")

[1/2] Downloading DocBank_500K_ori_img.zip.001...


✓ Part 1 downloaded
[2/2] Downloading DocBank_500K_ori_img.zip.002...
✓ Part 2 downloaded

Combining archives with zip -F...
Error: 

Trying alternative method with 7zip...



DocBank_500K_ori_img.zip
ERRORS:
Unexpected end of archive




7-Zip [64] 17.05 : Copyright (c) 1999-2021 Igor Pavlov : 2017-08-28
p7zip Version 17.05 (locale=utf8,Utf16=on,HugeFiles=on,64 bits,12 CPUs LE)

Scanning the drive for archives:
1 file, 5368709120 bytes (5120 MiB)

Extracting archive: DocBank_500K_ori_img.zip.001
--
Path = DocBank_500K_ori_img.zip.001
Type = Split
Physical Size = 5368709120
Volumes = 2
Total Physical Size = 10737418240
----
Path = DocBank_500K_ori_img.zip
Size = 10737418240
--
Path = DocBank_500K_ori_img.zip
Type = zip
ERRORS:
Unexpected end of archive
Physical Size = 10737436814
Characteristics = Local


Sub items Errors: 1

Archives with Errors: 1

Open Errors: 1

Sub items Errors: 1


ERROR: Data Error : DocBank_500K_ori_img/144.tar_1805.09065.gz_gascom2018_localtime_5_ori.jpg



✓ SUCCESS! 105952 images extracted


In [ ]:
from huggingface_hub import hf_hub_download
from pathlib import Path
import glob
import subprocess

# # Download additional parts (example: parts 003-005)
# EXTRA_PARTS = [3, 4, 5]
# ARCHIVE_BASENAME = "DocBank_500K_ori_img.zip"

# print("Downloading additional parts...")
# extra_downloaded = []
# for part_idx in EXTRA_PARTS:
#     filename = f"{ARCHIVE_BASENAME}.{part_idx:03d}"
#     print(f"Downloading {filename}...")
#     part_path = hf_hub_download(
#         repo_id="liminghao1630/DocBank",
#         filename=filename,
#         repo_type="dataset",
#         local_dir="."
#     )
#     extra_downloaded.append(Path(part_path))
#     print(f"✓ Part {part_idx} downloaded")

# Combine all parts using zip -F command
print("\nCombining all archive parts with zip -F...")
all_parts = sorted(Path('.').glob(f"{ARCHIVE_BASENAME}.*"), key=lambda p: p.name)

if not all_parts:
    raise FileNotFoundError("No archive parts found in current directory")

print(f"Found {len(all_parts)} parts total")

# Use zip -F to fix/combine the split archive
result = subprocess.run(
    ["zip", "-F", f"{ARCHIVE_BASENAME}.001", "--out", "combined.zip"],
    capture_output=True,
    text=True
)

if result.returncode != 0:
    print("Error with zip -F:", result.stderr)
    print("\nFalling back to 7zip method...")
    
    # Install 7zip if needed
    print("Installing p7zip (if not already installed)...")
    subprocess.run(["brew", "install", "p7zip"], capture_output=True)
    
    # Extract with 7zip
    print("Extracting with 7zip...")
    result = subprocess.run(
        ["7z", "x", f"{ARCHIVE_BASENAME}.001", "-o./images/"],
        capture_output=True,
        text=True
    )
    print(result.stdout)
else:
    # Extract the combined zip
    print("Extracting combined.zip...")
    result = subprocess.run(
        ["unzip", "-q", "combined.zip", "-d", "images/"],
        capture_output=True,
        text=True
    )
    
    if result.returncode != 0:
        print("Extraction output:", result.stderr)
    else:
        print("✓ Extraction complete")

# Verify
imgs = glob.glob("images/**/*.jpg", recursive=True) + glob.glob("images/**/*.png", recursive=True)
print(f"\n✓ Total images now: {len(imgs)}")

# Optional: Clean up combined.zip to save space
# Path("combined.zip").unlink(missing_ok=True)

# from huggingface_hub import hf_hub_download
# from pathlib import Path
# import glob

# # Download additional parts (example: parts 003-005)
# EXTRA_PARTS = [3, 4, 5]
# ARCHIVE_BASENAME = "DocBank_500K_ori_img.zip"

# extra_downloaded = []
# for part_idx in EXTRA_PARTS:
#     filename = f"{ARCHIVE_BASENAME}.{part_idx:03d}"
#     print(f"Downloading {filename}...")
#     part_path = hf_hub_download(
#         repo_id="liminghao1630/DocBank",
#         filename=filename,
#         repo_type="dataset",
#         local_dir=".",
#         resume_download=True,
#     )
#     extra_downloaded.append(Path(part_path))

# print("\nRebuilding archive from all downloaded parts and re-extracting...")
# all_parts = sorted(Path('.').glob(f"{ARCHIVE_BASENAME}.*"), key=lambda p: p.name)

# if not all_parts:
#     raise FileNotFoundError("No archive parts found in current directory")
# if "combine_split_zip" not in globals() or "extract_zip" not in globals():
#     raise RuntimeError("Run the previous download/extract cell first to define helper functions")

# combine_split_zip(all_parts, Path(ARCHIVE_BASENAME))
# extract_zip(Path(ARCHIVE_BASENAME), Path("images"))

# # Verify
# imgs = glob.glob("images/**/*.jpg", recursive=True) + glob.glob("images/**/*.png", recursive=True)
# print(f"Total images now: {len(imgs)}")


Combining all archive parts with zip -F...
Found 5 parts total
Error with zip -F: 

Falling back to 7zip method...
Installing p7zip (if not already installed)...
Extracting with 7zip...


## Step 4: Verify Downloaded Files

In [ ]:
import os
from pathlib import Path

print("File structure:")
print("="*50)

# # Check annotations
# ann_path = Path('annotations')
# if ann_path.exists():
#     json_files = list(ann_path.glob('*.json'))
#     print(f"\n✓ Annotations: {len(json_files)} JSON files")
#     for f in json_files:
#         size_mb = f.stat().st_size / (1024*1024)
#         print(f"  - {f.name}: {size_mb:.1f} MB")
# else:
#     print("\n✗ Annotations not found")

# Check images
img_path = Path('images')
if img_path.exists():
    img_files = list(img_path.rglob('*.jpg')) + list(img_path.rglob('*.png'))
    print(f"\n✓ Images: {len(img_files)} files")

    # Calculate total size
    total_size = sum(f.stat().st_size for f in img_files) / (1024**3)
    print(f"  Total size: {total_size:.2f} GB")
else:
    print("\n✗ Images not found")

File structure:

✓ Images: 27793 files
  Total size: 2.79 GB


In [ ]:
import json
from pathlib import Path
from collections import defaultdict

# Load all annotation files
annotations = {
    'train': json.load(open('annotations/500K_train.json')),
    'valid': json.load(open('annotations/500K_valid.json')),
    'test': json.load(open('annotations/500K_test.json'))
}

# Get all available image filenames
available_images = set()
for img_path in Path('images').rglob('*.jpg'):
    available_images.add(img_path.name)

print(f"Total available images: {len(available_images)}")
print("="*60)

# Check which images are referenced in each split
for split_name, data in annotations.items():
    total_refs = len(data['images'])
    filenames = [img['file_name'] for img in data['images']]

    # Count how many are available
    available_count = sum(1 for fn in filenames if fn in available_images)
    missing_count = total_refs - available_count

    print(f"\n{split_name.upper()}:")
    print(f"  Referenced in JSON: {total_refs}")
    print(f"  Available locally: {available_count}")
    print(f"  Missing: {missing_count}")
    print(f"  Coverage: {available_count/total_refs*100:.1f}%")

    # Show sample of what's missing
    if missing_count > 0:
        missing_samples = [fn for fn in filenames if fn not in available_images][:5]
        print(f"  Sample missing files:")
        for fn in missing_samples:
            print(f"    - {fn}")

# Check if images are organized in subdirectories
print("\n" + "="*60)
print("Image directory structure:")
for item in Path('images').iterdir():
    if item.is_dir():
        img_count = len(list(item.rglob('*.jpg')))
        print(f"  📁 {item.name}/ ({img_count} images)")

Total available images: 27793

TRAIN:
  Referenced in JSON: 400000
  Available locally: 22188
  Missing: 377812
  Coverage: 5.5%
  Sample missing files:
    - 121.tar_1706.01466.gz_Renormalization_Yukawa_FinV_2_ori.jpg
    - 237.tar_1612.01125.gz_testbeam_7_ori.jpg
    - 88.tar_1804.00089.gz_ms_10_ori.jpg
    - 210.tar_1807.08600.gz_X-ray_spectral_variability_of_blazars_using_principal_component_analysis_0_ori.jpg
    - 195.tar_1610.03254.gz_finite_beta_ver2.1_19_ori.jpg

VALID:
  Referenced in JSON: 50000
  Available locally: 2818
  Missing: 47182
  Coverage: 5.6%
  Sample missing files:
    - 105.tar_1705.06484.gz_Beck_s_Thm_Generalized_ArXiv_Version_8_ori.jpg
    - 105.tar_1705.06493.gz_gauge_4_ori.jpg
    - 105.tar_1705.06499.gz_NAUM_20170512_0_ori.jpg
    - 105.tar_1804.06048.gz_lecture_notes_11_ori.jpg
    - 105.tar_1804.06052.gz_An_Algorithm_for_the_Classification_of_Twisted_Forms_of_Toric_Varieties_7_ori.jpg

TEST:
  Referenced in JSON: 50000
  Available locally: 2787
  Missing

## Step 5: Convert to YOLO Format

Converting training set...
Converting train set from COCO format...


KeyboardInterrupt: 

In [ ]:
import json
import shutil
from pathlib import Path
from tqdm import tqdm

class DocBankToYOLO:
    def __init__(self, output_dir='yolo_dataset'):
        self.output_dir = Path(output_dir)

        # Map DocBank's 12 classes to 3 YOLO classes
        self.class_mapping = {
            'abstract': 0, 'author': 0, 'caption': 0, 'footer': 0,
            'list': 0, 'paragraph': 0, 'reference': 0, 'section': 0, 'title': 0,
            'equation': 1,  # formula
            'figure': 2, 'table': 2,  # image
        }

        self.class_names = ['text', 'formula', 'image']

        # Create directory structure
        for split in ['train', 'val', 'test']:
            (self.output_dir / 'images' / split).mkdir(parents=True, exist_ok=True)
            (self.output_dir / 'labels' / split).mkdir(parents=True, exist_ok=True)

    def convert_from_coco(self, coco_json_path, image_dir, split='train', limit=None):
        """
        Convert from MSCOCO format JSON

        Args:
            coco_json_path: Path to COCO format JSON file
            image_dir: Directory containing images
            split: 'train', 'val', or 'test'
            limit: Maximum number of images to process (for testing)
        """
        print(f"Converting {split} set from COCO format...")

        with open(coco_json_path) as f:
            coco = json.load(f)

        # Create mappings
        images_info = {
            img['id']: {
                'filename': img['file_name'],
                'width': img['width'],
                'height': img['height']
            }
            for img in coco['images']
        }

        categories = {
            cat['id']: cat['name'].lower()
            for cat in coco['categories']
        }

        # Group annotations by image
        img_annotations = {}
        for ann in coco['annotations']:
            img_id = ann['image_id']
            if img_id not in img_annotations:
                img_annotations[img_id] = []
            img_annotations[img_id].append(ann)

        # Apply limit if specified
        if limit:
            img_annotations = dict(list(img_annotations.items())[:limit])

        # Process each image
        processed = 0
        skipped = 0

        for img_id, annotations in tqdm(img_annotations.items(), desc=f"Processing {split}"):
            img_info = images_info[img_id]
            img_filename = img_info['filename']
            img_width = img_info['width']
            img_height = img_info['height']

            # Find image file
            src_img_path = Path(image_dir) / img_filename
            if not src_img_path.exists():
                # Try to find in subdirectories
                found = list(Path(image_dir).rglob(img_filename))
                if found:
                    src_img_path = found[0]
                else:
                    skipped += 1
                    continue

            # Copy image
            dst_img_path = self.output_dir / 'images' / split / img_filename
            shutil.copy(src_img_path, dst_img_path)

            # Convert annotations to YOLO format
            yolo_lines = []
            for ann in annotations:
                cat_name = categories[ann['category_id']]

                if cat_name not in self.class_mapping:
                    continue

                class_id = self.class_mapping[cat_name]
                x_min, y_min, bbox_w, bbox_h = ann['bbox']

                # Convert to YOLO format
                x_center = (x_min + bbox_w / 2) / img_width
                y_center = (y_min + bbox_h / 2) / img_height
                norm_width = bbox_w / img_width
                norm_height = bbox_h / img_height

                # Clamp to valid range
                x_center = max(0, min(1, x_center))
                y_center = max(0, min(1, y_center))
                norm_width = max(0, min(1, norm_width))
                norm_height = max(0, min(1, norm_height))

                yolo_lines.append(
                    f"{class_id} {x_center:.6f} {y_center:.6f} "
                    f"{norm_width:.6f} {norm_height:.6f}"
                )

            # Save labels
            label_path = self.output_dir / 'labels' / split / f"{Path(img_filename).stem}.txt"
            label_path.write_text('\n'.join(yolo_lines))

            processed += 1

        print(f"✓ Processed: {processed}, Skipped: {skipped}")
        return processed, skipped

    def create_yaml(self):
        """Create data.yaml for YOLO training"""
        yaml_content = f"""# DocBank Dataset for YOLO
path: {self.output_dir.absolute()}
train: images/train
val: images/val
test: images/test

# Classes
nc: 3
names: {self.class_names}
"""

        yaml_path = self.output_dir / 'data.yaml'
        yaml_path.write_text(yaml_content)
        print(f"✓ Created {yaml_path}")

    def print_statistics(self):
        """Print dataset statistics"""
        print("\n" + "="*50)
        print("Dataset Statistics")
        print("="*50)

        for split in ['train', 'val', 'test']:
            img_dir = self.output_dir / 'images' / split
            label_dir = self.output_dir / 'labels' / split

            num_images = len(list(img_dir.glob('*')))
            num_labels = len(list(label_dir.glob('*.txt')))

            # Count instances per class
            class_counts = {i: 0 for i in range(3)}
            for label_file in label_dir.glob('*.txt'):
                with open(label_file) as f:
                    for line in f:
                        if line.strip():
                            class_id = int(line.split()[0])
                            class_counts[class_id] += 1

            print(f"\n{split.upper()}:")
            print(f"  Images: {num_images}")
            print(f"  Labels: {num_labels}")
            print(f"  Instances:")
            for class_id, count in class_counts.items():
                print(f"    {self.class_names[class_id]}: {count}")

# Create converter instance
converter = DocBankToYOLO(output_dir='docbank_yolo')
print("✓ Converter initialized")

✓ Converter initialized


In [ ]:
# # Convert all splits
# # For testing with subset, add limit parameter: limit=1000

# print("Converting training set...")
# converter.convert_from_coco(
#     coco_json_path='annotations/train.json',
#     image_dir='images',
#     split='train',
#     # limit=1000  # Uncomment to process only first 1000 images for testing
# )

# print("\nConverting validation set...")
# converter.convert_from_coco(
#     coco_json_path='annotations/val.json',
#     image_dir='images',
#     split='val',
#     # limit=200
# )

# print("\nConverting test set...")
# converter.convert_from_coco(
#     coco_json_path='annotations/test.json',
#     image_dir='images',
#     split='test',
#     # limit=200
# )

# # Create YAML config
# converter.create_yaml()

# # Print statistics
# converter.print_statistics()

Converting training set...
Converting train set from COCO format...


FileNotFoundError: [Errno 2] No such file or directory: 'annotations/train.json'

In [ ]:
# # Convert training set (start with 1000 images for testing)
# print("Converting training set...")
# converter.convert_from_coco(
#     coco_json_path='annotations/500K_train.json',  # Note: 500K_train not train
#     image_dir='images',
#     split='train',
#     limit=2700  # Testing with 1000 images first
# )

# Convert validation set
print("\nConverting validation set...")
converter.convert_from_coco(
    coco_json_path='annotations/500K_valid.json',  # Note: 500K_valid not val
    image_dir='images',
    split='val',
    limit=1000
)

# Convert test set
# print("\nConverting test set...")
# converter.convert_from_coco(
#     coco_json_path='annotations/500K_test.json',  # Note: 500K_test not test
#     image_dir='images',
#     split='test',
#     limit=100
# )

# Create YAML config
converter.create_yaml()

# Print statistics
converter.print_statistics()


Converting test set...
Converting test set from COCO format...


Processing test: 100%|██████████| 100/100 [44:57<00:00, 26.97s/it]


✓ Processed: 6, Skipped: 94
✓ Created docbank_yolo/data.yaml

Dataset Statistics

TRAIN:
  Images: 170
  Labels: 170
  Instances:
    text: 1758
    formula: 346
    image: 64

VAL:
  Images: 1000
  Labels: 1000
  Instances:
    text: 10406
    formula: 2499
    image: 401

TEST:
  Images: 20
  Labels: 20
  Instances:
    text: 223
    formula: 59
    image: 3


## Step 6: Train YOLO Model

In [ ]:
from ultralytics import YOLO

# Initialize model
model = YOLO('yolov8n.pt')  # Use yolov8s.pt or yolov8m.pt for better accuracy

# Train
results = model.train(
    data='docbank_yolo/data.yaml',
    epochs=100,
    imgsz=640,
    batch=16,  # Adjust based on GPU memory
    device=0,  # Use GPU
    workers=2,
    patience=10,
    save=True,
    project='docbank_training',
    name='yolov8n_docbank',

    # Data augmentation
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=5.0,
    translate=0.1,
    scale=0.5,
    shear=2.0,
    perspective=0.0,
    flipud=0.0,
    fliplr=0.5,
    mosaic=1.0,
    mixup=0.0,
)

print("\n✓ Training complete!")
print(f"Best weights saved to: {results.save_dir}/weights/best.pt")

## Step 7: Validate Model

In [ ]:
# Load best model
best_model = YOLO('docbank_training/yolov8n_docbank/weights/best.pt')

# Validate
metrics = best_model.val(
    data='docbank_yolo/data.yaml',
    split='val'
)

print("\nValidation Results:")
print("="*50)
print(f"mAP50: {metrics.box.map50:.3f}")
print(f"mAP50-95: {metrics.box.map:.3f}")
print("\nPer-class AP50:")
for i, ap in enumerate(metrics.box.ap50):
    print(f"  {converter.class_names[i]}: {ap:.3f}")

## Step 8: Test Inference

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt
from pathlib import Path

# Get a test image
test_images = list(Path('docbank_yolo/images/test').glob('*.jpg'))[:5]

# Run inference
results = best_model.predict(
    source=test_images,
    conf=0.25,
    save=True,
    project='inference_results',
    name='test_predictions'
)

# Display results
print(f"\nResults saved to: inference_results/test_predictions/")
print("\nShowing first prediction:")

# Display first result
img = Image.open(f'inference_results/test_predictions/{test_images[0].name}')
plt.figure(figsize=(12, 8))
plt.imshow(img)
plt.axis('off')
plt.title('Detection Results: Blue=Text, Orange=Formula, Green=Image')
plt.show()

# Print detection counts
for i, result in enumerate(results):
    boxes = result.boxes
    print(f"\nImage {i+1}: {len(boxes)} detections")
    class_counts = {}
    for box in boxes:
        cls = int(box.cls[0])
        class_name = converter.class_names[cls]
        class_counts[class_name] = class_counts.get(class_name, 0) + 1
    for cls_name, count in class_counts.items():
        print(f"  {cls_name}: {count}")

## Optional: Export Model

In [ ]:
# Export to ONNX for deployment
best_model.export(format='onnx')
print("✓ Model exported to ONNX format")

# Export to TensorRT for faster inference (requires CUDA)
# best_model.export(format='engine')

# Export to CoreML for iOS
# best_model.export(format='coreml')

## Tips for Better Results

1. **Use larger model**: Try `yolov8m.pt` or `yolov8l.pt` for better accuracy
2. **Increase image size**: Use `imgsz=1024` if GPU memory allows
3. **Class imbalance**: Formulas are rare - consider:
   - Increasing formula samples via augmentation
   - Using weighted loss
   - Oversampling formula-heavy documents
4. **Longer training**: Full convergence may need 200+ epochs
5. **Mixed datasets**: Combine DocBank with PubLayNet for better generalization